# 04 — Generation evaluation (end-to-end через agent-эндпоинт)

Для каждой коллекции:
1. Создаём тестового агента с инструментом `{collection_name}_search`
2. Прогоняем все вопросы через агента
3. Собираем traces (answer + retrieved_chunks + tool_calls)
4. Проверяем negative rejection на unanswerable вопросах
5. Опционально — RAGAS метрики

In [ ]:
import sys
import json
import time

sys.path.insert(0, "..")

from config import DATA_DIR, EVAL_DATASETS

In [ ]:
with open(DATA_DIR / "eval_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

collection_names = manifest["collections"]
dataset_keys = manifest["datasets"]
print(f"Manifest: {len(dataset_keys)} datasets, {len(collection_names)} collections")

## 1. Загрузка QA-вопросов (синтетические + native для BEIR/HotpotQA)

In [ ]:
from evalkit.dataset_loaders import (
    load_ir_dataset,
    load_hotpotqa,
    load_synthetic,
    load_rubq_retrieval,
)

# Load questions for EACH collection separately (one pass over collection_details)
collection_questions = {}  # collection_name -> list of questions
collection_to_dataset = {}  # collection_name -> dataset_key
total_questions = 0
total_answerable = 0
total_unanswerable = 0

for cd in manifest["collection_details"]:
    collection_name = cd["collection_name"]
    ds_key = cd["dataset_key"]
    collection_to_dataset[collection_name] = ds_key

    ds_cfg = EVAL_DATASETS.get(ds_key)
    if not ds_cfg:
        print(f"  SKIP {collection_name}: dataset {ds_key} not in EVAL_DATASETS")
        continue

    ds_type = ds_cfg["type"]
    if ds_type == "synthetic":
        _, qa_rows = load_synthetic(DATA_DIR / ds_cfg["path"])
    elif ds_type == "beir":
        _, qa_rows = load_ir_dataset(hf_name=ds_cfg["hf_name"])
    elif ds_type == "hotpotqa":
        _, qa_rows = load_hotpotqa(
            hf_name=ds_cfg["hf_name"], max_examples=ds_cfg.get("max_examples")
        )
    elif ds_type == "rubq":
        _, qa_rows = load_rubq_retrieval(hf_name=ds_cfg["hf_name"])
    else:
        print(f"  SKIP {collection_name}: unknown type {ds_type}")
        continue

    collection_questions[collection_name] = qa_rows
    n_ans = sum(1 for q in qa_rows if q.get("gold_spans"))
    n_unans = len(qa_rows) - n_ans
    total_questions += len(qa_rows)
    total_answerable += n_ans
    total_unanswerable += n_unans
    print(
        f"  {collection_name}: {len(qa_rows)} questions ({n_ans} answerable, {n_unans} unanswerable)"
    )

print(
    f"\nTotal: {total_questions} questions ({total_answerable} answerable, {total_unanswerable} unanswerable)"
)

## 2–3. Тестовые агенты и прогон вопросов

Для каждой коллекции создаём агента (`ensure_eval_agent`), прогоняем через него
все вопросы и сохраняем traces в `generation_traces.jsonl`.

Поиск и ответы выполняются агентами agent-core. Настройки загружаются из JSON.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from config import AGENT_WORKER_POOL_SIZE
from evalkit.client import make_client

client = make_client()


def run_one(q, collection_name, agent_id):
    try:
        res = client.agent_query(q["question"], agent_id=agent_id)
    except Exception as e:  # noqa: BLE001 — preserve independent question failures
        res = {
            "answer": "",
            "tool_calls": [],
            "retrieved_chunks": [],
            "trace_id": "",
            "error": str(e),
        }
    return {
        "collection": collection_name,
        "qid": q.get("qid"),
        "question": q["question"],
        "question_type": q.get("question_type"),
        "dataset_source": q.get(
            "dataset_source", collection_to_dataset[collection_name]
        ),
        "reference_answer": q.get("reference_answer") or "",
        **res,
    }


traces = []
t0 = time.perf_counter()
for collection_name, qa_rows in collection_questions.items():
    agent_id = client.ensure_eval_agent(collection_name)
    print(f"{collection_name}: agent_id={agent_id}, {len(qa_rows)} questions")
    with ThreadPoolExecutor(max_workers=AGENT_WORKER_POOL_SIZE) as pool:
        futures = [pool.submit(run_one, q, collection_name, agent_id) for q in qa_rows]
        for fut in tqdm(
            as_completed(futures), total=len(futures), desc=collection_name, unit="q"
        ):
            traces.append(fut.result())
traces_path = DATA_DIR / "generation_traces.jsonl"
with open(traces_path, "w", encoding="utf-8") as f:
    f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in traces)
n_err = sum(1 for t in traces if t.get("error"))
print(
    f"\nSaved {len(traces)} traces -> {traces_path} ({time.perf_counter() - t0:.1f}s)"
)
if n_err:
    print(
        f"WARN: {n_err} queries failed, e.g.: {next(t['error'] for t in traces if t.get('error'))}"
    )

## 4. Evaluation (negative rejection + RAGAS)

Запускает внешний скрипт `notebooks/eval_direct.py`:
- Section 4: Negative rejection check (unanswerable questions)
- Section 5: DeepEval metrics evaluation (manual parallel)

In [ ]:
import subprocess
import sys
from pathlib import Path

# Run evaluation script
script_path = Path("../notebooks/eval_direct.py")
result = subprocess.run(
    [sys.executable, str(script_path)],
    check=False,
    capture_output=True,
    text=True,
)

print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print(f"\nExit code: {result.returncode}")
if result.returncode != 0:
    sys.exit(result.returncode)